# ✋ Hand Gesture Recognition System
### Major Project — Computer Vision & Deep Learning

---

**Tech Stack:** `OpenCV` · `CVZone` · `Keras / TensorFlow` · `Teachable Machine`

**Supported Gestures:** Hello · I Love You · No · Okay · Please · Thank You · Yes

---

## 📦 1. Install Dependencies

In [4]:
# Run once to install all required packages
!pip install opencv-python cvzone tensorflow streamlit numpy Pillow
!pip install mediapipe==0.10.14

## 📂 2. Project Structure

In [5]:
import os

GESTURE_LABELS = ["Hello", "I Love You", "No", "Okay", "Please", "Thank You", "Yes"]

# Create folder structure
for label in GESTURE_LABELS:
    os.makedirs(f"Data/{label}", exist_ok=True)

os.makedirs("model", exist_ok=True)

print("✅ Project structure ready!")
print("\nFolders created:")
for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = " " * 3 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files:
            print(f"{indent}   {f}")

✅ Project structure ready!

Folders created:
./
   .config/
      .last_update_check.json
      active_config
      .last_survey_prompt.yaml
      hidden_gcloud_config_universe_descriptor_data_cache_configs.db
      .last_opt_in_prompt.yaml
      default_configs.db
      gce
      config_sentinel
      logs/
         2026.04.02/
      configurations/
   Data/
      No/
      Yes/
      Okay/
      Thank You/
      Please/
      I Love You/
      Hello/
   model/
   sample_data/
      README.md
      anscombe.json
      mnist_test.csv
      california_housing_train.csv
      california_housing_test.csv
      mnist_train_small.csv


## 📸 3. Data Collection

> Collect training images for each gesture.
> **Set the `GESTURE` variable**, then run the cell. Press `S` to save, `Q` to quit.

In [3]:
import cv2
from cvzone.HandTrackingModule import HandDetector
import numpy as np
import math
import time

# ──────────────────────────────────────────
GESTURE = "Hello"   # ← Change for each class
TARGET  = 300        # images to collect
# ──────────────────────────────────────────

OFFSET   = 20
IMG_SIZE = 300
folder   = f"Data/{GESTURE}"
os.makedirs(folder, exist_ok=True)

cap      = cv2.VideoCapture(0)
detector = HandDetector(maxHands=1)
counter  = 0

print(f"Collecting [{GESTURE}] — Press S to save, Q to quit")

def safe_crop(img, y, h, x, w, offset):
    H, W = img.shape[:2]
    return img[max(0,y-offset):min(H,y+h+offset), max(0,x-offset):min(W,x+w+offset)]

def to_canvas(crop):
    c = np.ones((IMG_SIZE, IMG_SIZE, 3), np.uint8) * 255
    h, w = crop.shape[:2]
    if h/w > 1:
        wc = math.ceil(IMG_SIZE/h * w)
        wg = math.ceil((IMG_SIZE-wc)/2)
        c[:, wg:wc+wg] = cv2.resize(crop, (wc, IMG_SIZE))
    else:
        hc = math.ceil(IMG_SIZE/w * h)
        hg = math.ceil((IMG_SIZE-hc)/2)
        c[hg:hc+hg, :] = cv2.resize(crop, (IMG_SIZE, hc))
    return c

while True:
    success, img = cap.read()
    hands, _ = detector.findHands(img)
    crop = None

    if hands:
        x, y, w, h = hands[0]["bbox"]
        crop = safe_crop(img, y, h, x, w, OFFSET)
        if crop.size > 0:
            canvas = to_canvas(crop)
            cv2.imshow("Canvas", canvas)
        cv2.rectangle(img, (x-OFFSET, y-OFFSET),
                      (x+w+OFFSET, y+h+OFFSET), (0, 229, 255), 2)

    prog = int((counter/TARGET)*img.shape[1])
    cv2.rectangle(img, (0,0), (prog, 8), (0, 229, 255), -1)
    cv2.putText(img, f"{GESTURE}: {counter}/{TARGET}",
                (15, 35), cv2.FONT_HERSHEY_SIMPLEX, .9, (0,229,255), 2)
    cv2.imshow("Collect (S=save, Q=quit)", img)

    key = cv2.waitKey(1)
    if key == ord("s") and crop is not None and crop.size > 0:
        cv2.imwrite(f"{folder}/img_{int(time.time()*1000)}.jpg", canvas)
        counter += 1
        if counter % 50 == 0:
            print(f"  {counter}/{TARGET} saved…")
        if counter >= TARGET:
            print(f"✅ Done collecting {TARGET} images for [{GESTURE}]")
            break
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Collected {counter} images for [{GESTURE}]")

NameError: name 'os' is not defined

## 🔢 4. Inspect Dataset

In [4]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

counts = {}
for label in GESTURE_LABELS:
    path = f"Data/{label}"
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    counts[label] = n

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor("#080b12")

ax = axes[0]
ax.set_facecolor("#0f1420")
bars = ax.bar(counts.keys(), counts.values(),
               color=["#00e5ff","#7b61ff","#00ffb3","#ffb700",
                       "#ff4566","#00e5ff","#7b61ff"],
               edgecolor="#1e2640", linewidth=1.2)
ax.set_title("Images per Gesture Class", color="#e8eaf6", fontsize=13, pad=10)
ax.set_xlabel("Gesture", color="#5c6080")
ax.set_ylabel("Count",   color="#5c6080")
ax.tick_params(colors="#5c6080")
for spine in ax.spines.values(): spine.set_edgecolor("#1e2640")
for bar, val in zip(bars, counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha="center", va="bottom", color="#e8eaf6", fontsize=9)

# Sample images
ax2 = axes[1]
ax2.axis("off")
ax2.set_facecolor("#080b12")
sample_text = "\n".join([f"  {k:12s}  →  {v:>4d} images" for k, v in counts.items()])
ax2.text(0.1, 0.5, sample_text, transform=ax2.transAxes,
         fontsize=11, verticalalignment="center",
         fontfamily="monospace", color="#00e5ff",
         bbox=dict(boxstyle="round", facecolor="#0f1420", edgecolor="#1e2640"))
ax2.set_title("Dataset Summary", color="#e8eaf6", fontsize=13)

plt.tight_layout()
plt.savefig("dataset_summary.png", dpi=120, bbox_inches="tight",
            facecolor="#080b12")
plt.show()
print(f"Total images: {sum(counts.values())}")

NameError: name 'GESTURE_LABELS' is not defined

## 🤖 5. Train Model — Google Teachable Machine

After collecting data, train your model using **[Teachable Machine](https://teachablemachine.withgoogle.com/)**:

1. Go to → https://teachablemachine.withgoogle.com/
2. Choose **Image Project → Standard image model**
3. Create one class per gesture, upload images from your `Data/` folders
4. Click **Train Model**
5. Export → **Tensorflow** → Keras (.h5)
6. Download and place files in the `model/` folder:
   - `model/keras_model.h5`
   - `model/labels.txt`

## 🔬 6. Test Model (Single Frame)

In [5]:
import cv2
import numpy as np
import math
from cvzone.HandTrackingModule import HandDetector
from cvzone.ClassificationModule import Classifier
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

OFFSET   = 20
IMG_SIZE = 300
CONFIDENCE_THRESHOLD = 0.60

detector   = HandDetector(maxHands=1)
classifier = Classifier("model/keras_model.h5", "model/labels.txt")

with open("model/labels.txt") as f:
    labels = [l.strip() for l in f if l.strip()]

def predict_gesture(frame_rgb):
    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    hands, _  = detector.findHands(frame_bgr, draw=False)
    if not hands:
        return None, 0.0, None
    x, y, w, h = hands[0]["bbox"]
    H, W = frame_bgr.shape[:2]
    crop = frame_bgr[max(0,y-OFFSET):min(H,y+h+OFFSET), max(0,x-OFFSET):min(W,x+w+OFFSET)]
    if crop.size == 0:
        return None, 0.0, None
    canvas = np.ones((IMG_SIZE, IMG_SIZE, 3), np.uint8) * 255
    ratio  = h / w
    if ratio > 1:
        wc = math.ceil(IMG_SIZE/h*w); wg = math.ceil((IMG_SIZE-wc)/2)
        canvas[:, wg:wc+wg] = cv2.resize(crop, (wc, IMG_SIZE))
    else:
        hc = math.ceil(IMG_SIZE/w*h); hg = math.ceil((IMG_SIZE-hc)/2)
        canvas[hg:hc+hg, :] = cv2.resize(crop, (IMG_SIZE, hc))
    prediction, index = classifier.getPrediction(canvas, draw=False)
    conf = float(max(prediction))
    return labels[index] if conf >= CONFIDENCE_THRESHOLD else "Uncertain", conf, canvas

# Capture one frame and visualise
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    label, conf, canvas = predict_gesture(frame_rgb)

    fig, axes = plt.subplots(1, 2 if canvas is not None else 1, figsize=(12, 5))
    fig.patch.set_facecolor("#080b12")
    if canvas is None:
        axes = [axes]

    axes[0].imshow(frame_rgb)
    axes[0].set_title(f"Camera Frame", color="#e8eaf6", fontsize=12)
    axes[0].axis("off")
    axes[0].set_facecolor("#080b12")

    if canvas is not None:
        axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Model Input Canvas\nPrediction: {label}  ({conf*100:.1f}%)",
                          color="#00e5ff", fontsize=12)
        axes[1].axis("off")
        axes[1].set_facecolor("#080b12")

    fig.suptitle(f"Predicted Gesture: {label or 'No Hand'}",
                 color="#ffffff", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"\nResult: {label}  |  Confidence: {conf*100:.1f}%")
else:
    print("❌ Could not read from camera.")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'model/keras_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## 🎥 7. Real-Time Recognition (Inline in Notebook)

In [ ]:
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import PIL.Image
import io
import time

# UI widgets
img_widget   = widgets.Image(format="jpeg", width=640, height=480)
label_widget = widgets.HTML(value="<h2 style='color:#00e5ff; font-family:monospace'>Waiting…</h2>")
conf_widget  = widgets.FloatProgress(value=0, min=0, max=1,
                                       description="Confidence:",
                                       bar_style="info",
                                       style={"bar_color": "#00e5ff"},
                                       layout=widgets.Layout(width="400px"))
stop_btn = widgets.Button(description="⏹ Stop",
                           button_style="danger",
                           layout=widgets.Layout(width="120px"))
status_lbl = widgets.Label("Status: running")

display(widgets.VBox([
    widgets.HBox([img_widget, widgets.VBox([label_widget, conf_widget, status_lbl])]),
    stop_btn
]))

running = [True]
def on_stop(b):
    running[0] = False
    status_lbl.value = "Status: stopped"
stop_btn.on_click(on_stop)

def frame_to_jpeg(frame_rgb):
    pil_img = PIL.Image.fromarray(frame_rgb)
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG", quality=75)
    return buf.getvalue()

cap = cv2.VideoCapture(0)

while running[0]:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    label, conf, canvas = predict_gesture(frame_rgb)

    # Draw on frame
    txt   = f"{label}  {int(conf*100)}%" if label else "No Hand Detected"
    color = (0, 229, 255) if label and label not in ("Uncertain", None) else (100, 100, 100)
    cv2.putText(frame_rgb, txt, (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    img_widget.value   = frame_to_jpeg(frame_rgb)
    label_widget.value = f"<h2 style='color:#00e5ff; font-family:monospace'>{txt}</h2>"
    conf_widget.value  = conf

    time.sleep(0.04)

cap.release()
print("Camera released.")

## 🌐 8. Launch Full Streamlit App

In [6]:
# Run the full Streamlit app from the notebook
# Opens in a new browser tab
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print("✅ Streamlit app launched! Open http://localhost:8501 in your browser.")
print("   Run proc.terminate() to stop it.")

✅ Streamlit app launched! Open http://localhost:8501 in your browser.
   Run proc.terminate() to stop it.


## 📊 9. Confusion Matrix (Post-Training Evaluation)

In [7]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Evaluate on a random sample from each class
SAMPLE_PER_CLASS = 30

y_true, y_pred = [], []

for idx, label in enumerate(labels):
    folder = f"Data/{label}"
    if not os.path.exists(folder):
        continue
    files = [f for f in os.listdir(folder) if f.endswith(".jpg")]
    sample = random.sample(files, min(SAMPLE_PER_CLASS, len(files)))
    for fname in sample:
        img = cv2.imread(os.path.join(folder, fname))
        if img is None:
            continue
        pred, _, _ = classifier.getPrediction(img, draw=False)
        y_true.append(idx)
        y_pred.append(int(np.argmax(pred)))

# Confusion matrix
n = len(labels)
cm = np.zeros((n, n), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t][p] += 1

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor("#080b12")
ax.set_facecolor("#0f1420")

im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=35, ha="right", color="#e8eaf6")
ax.set_yticks(range(n)); ax.set_yticklabels(labels, color="#e8eaf6")
ax.set_xlabel("Predicted", color="#5c6080", labelpad=10)
ax.set_ylabel("True Label", color="#5c6080", labelpad=10)
ax.set_title("Confusion Matrix", color="#e8eaf6", fontsize=14, pad=15)

for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="#00e5ff" if cm[i,j] > cm.max()/2 else "#e8eaf6",
                fontsize=11, fontweight="bold")

for spine in ax.spines.values(): spine.set_edgecolor("#1e2640")
plt.colorbar(im, ax=ax).ax.yaxis.set_tick_params(color="#5c6080")

acc = np.trace(cm) / cm.sum() * 100
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches="tight", facecolor="#080b12")
plt.show()
print(f"\nOverall Accuracy: {acc:.1f}%")

NameError: name 'labels' is not defined

---

## 🎓 Project Summary

| Component | Technology |
|:---|:---|
| Hand Detection | CVZone `HandDetector` (MediaPipe) |
| Classification | Keras `.h5` from Teachable Machine |
| Preprocessing | OpenCV — aspect-ratio canvas resize |
| UI (Web App) | Streamlit |
| Notebook UI | ipywidgets + matplotlib |
| Gestures | Hello · I Love You · No · Okay · Please · Thank You · Yes |

---
*Made with ❤️ for Major Project Presentation*